# 🧠 Member 4 — T2f (FLAIR) Pipeline
**BraTS GLI-2024 | ANN Course Project**

| Cell | What it does |
|------|-------------|
| 1 | Install deps + Setup paths |
| 2 | Sanity check — all 4 modalities + tumour overlay |
| 3 | Model check — both architectures |
| 4 | Dataset check — patch shapes + tumour-bias sampling |
| 5 | **Train both models** (ResUNet3D + SegResNet) — MLflow logged |
| 6 | **Evaluate** on test set — Dice / IoU / HD95 |
| 7 | **Grad-CAM XAI** — 3-view overlays + arch comparison |
| 8 | **Launch Gradio app** — public share link |

> Run **Shift+Enter** cell by cell. Every cell ends with a visual check.

---
## Cell 1 — Install dependencies & setup

In [ ]:
# ── Install (Kaggle pre-installs torch/monai, but just in case) ───────────────
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

for pkg in ['nibabel', 'monai', 'mlflow', 'gradio', 'shap', 'scipy', 'tqdm']:
    pip_install(pkg)

print('All packages ready')

In [ ]:
import os, sys
from pathlib import Path

# ── Detect environment ───────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/working')
print(f'Running on Kaggle: {IS_KAGGLE}')

if IS_KAGGLE:
    # Add shared infra from your Kaggle dataset
    # Upload your 'brats-code' dataset containing the shared/ folder
    for candidate in [
        '/kaggle/input/brats-code/shared',
        '/kaggle/input/brats-shared-infra',
    ]:
        parent = str(Path(candidate).parent)
        if Path(candidate).exists() and parent not in sys.path:
            sys.path.insert(0, parent)
            print(f'Added to path: {parent}')
            break
    REPO_ROOT    = Path('/kaggle/working')
    RESULTS_ROOT = Path('/kaggle/working/results')
    # M4 code is in the working directory (uploaded alongside notebook)
    if str(REPO_ROOT / 'member4_T2f') not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / 'member4_T2f'))
else:
    REPO_ROOT    = Path().resolve().parent
    RESULTS_ROOT = REPO_ROOT / 'results'
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

# ── Core imports ──────────────────────────────────────────────────────────────
from shared.seed import set_global_seed
set_global_seed()   # seed = 42, always first

from shared.config import get_data_root, SPLITS_DIR, PATCH_SIZE, CHECKPOINT_DIR
from shared.dataset import BraTSDataset, get_dataloader
from shared.metrics import compute_all_metrics
from shared.preprocessing import preprocess_patient

import torch
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DATA_ROOT  = get_data_root()
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PATCH_SIZE_VAL = PATCH_SIZE[0]

# Print environment summary
print(f'\nRepo root   : {REPO_ROOT}')
print(f'Data root   : {DATA_ROOT}')
print(f'Results dir : {RESULTS_ROOT}')
print(f'Device      : {DEVICE}')
if DEVICE.type == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU         : {torch.cuda.get_device_name(0)} | {vram:.1f} GB')
print(f'Patch size  : {PATCH_SIZE_VAL}')
print('\n✓ Setup OK')

---
## Cell 2 — Sanity check: all 4 modalities + tumour overlay

In [ ]:
import glob, os
from shared.preprocessing import load_and_normalise

# Pick the first patient
all_patients = sorted(os.listdir(DATA_ROOT))
PATIENT_ID   = all_patients[0]
PATIENT_DIR  = os.path.join(DATA_ROOT, PATIENT_ID)
print(f'Patient: {PATIENT_ID}')

def load_mod(mod):
    path = glob.glob(os.path.join(PATIENT_DIR, f'*{mod}*.nii*'))[0]
    return load_and_normalise(path)

vols    = {m: load_mod(m) for m in ['t1c', 't1n', 't2f', 't2w']}
seg_raw = nib.load(glob.glob(os.path.join(PATIENT_DIR, '*seg*.nii*'))[0]).get_fdata()

# Best axial slice = most tumour voxels
tumour_per_slice = (seg_raw > 0).sum(axis=(1, 2))
best_z = int(tumour_per_slice.argmax())
print(f'Best axial slice: z={best_z} ({tumour_per_slice[best_z]:.0f} tumour voxels)')

BG_COLOR = '#0D1117'
fig, axes = plt.subplots(2, 4, figsize=(20, 9), facecolor=BG_COLOR)
fig.suptitle(f'{PATIENT_ID} | axial z={best_z}\n'
             'TOP: raw modality slices | BOTTOM: with tumour contours',
             color='white', fontsize=12, y=0.98)

modalities  = ['t1c', 't1n', 't2f', 't2w']
titles      = ['T1c (contrast)', 'T1n (native)', 'T2f FLAIR ← YOUR modality', 'T2w']
TEAL        = '#5DCAA5'
AMBER       = '#EF9F27'

for col, (mod, title) in enumerate(zip(modalities, titles)):
    for row in range(2):
        ax = axes[row, col]
        ax.set_facecolor(BG_COLOR)
        ax.imshow(vols[mod][best_z].T, cmap='gray', origin='lower')
        if row == 1:  # add contours on bottom row
            for lbl, col_hex in [(2, TEAL), (4, AMBER)]:
                mask = (seg_raw[best_z] == lbl)
                if mask.any():
                    ax.contour(mask.T, levels=[0.5], colors=[col_hex], linewidths=1.8)
        if row == 0:
            ax.set_title(title, color='white', fontsize=9, pad=4)
        ax.axis('off')

# Add row labels
axes[0, 0].set_ylabel('Raw intensity', color='#aaa', fontsize=9, rotation=90)
axes[1, 0].set_ylabel('+ Tumour contours', color='#aaa', fontsize=9, rotation=90)

legend = [
    mpatches.Patch(color=TEAL,  label='Tumour core (label 2)'),
    mpatches.Patch(color=AMBER, label='Enhancing tumour (label 4)'),
]
axes[1, -1].legend(handles=legend, loc='lower right',
                   facecolor='#222', labelcolor='white', fontsize=8, framealpha=0.9)

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
(RESULTS_ROOT / 'T2F' / 'figures').mkdir(parents=True, exist_ok=True)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(RESULTS_ROOT / 'T2F' / 'figures' / 'M4_sanity.png',
            dpi=120, bbox_inches='tight', facecolor=BG_COLOR)
plt.show()

print('\nFLAIR note: CSF (ventricles) is DARK because the inversion pulse nulls it.')
print('Tumour and oedema stay BRIGHT — this is what your model learns to find.')

---
## Cell 3 — Model check: both architectures

In [ ]:
from model import build_model

dummy = torch.randn(1, 1, PATCH_SIZE_VAL, PATCH_SIZE_VAL, PATCH_SIZE_VAL)

rows = []
for arch in ('resunet', 'segresnet'):
    model  = build_model(arch, in_channels=1)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    with torch.no_grad():
        out        = model(dummy)
        out2, feat = model(dummy, return_features=True)
    assert out.shape == dummy.shape, f'Shape mismatch for {arch}!'
    rows.append({
        'arch':          model.arch_name,
        'params':        f'{params/1e6:.2f} M',
        'input_shape':   str(tuple(dummy.shape)),
        'output_shape':  str(tuple(out.shape)),
        'features_shape': str(tuple(feat.shape)) if feat is not None else 'N/A',
    })

import pandas as pd
df = pd.DataFrame(rows).set_index('arch')
print(df.to_string())
print('\n✓ Both models OK')

---
## Cell 4 — Dataset check

In [ ]:
train_ds = BraTSDataset(
    data_root          = DATA_ROOT,
    split_file         = str(SPLITS_DIR / 'train_ids.txt'),
    modality           = 't2f',
    patch_size         = PATCH_SIZE_VAL,
    augment            = False,
    patches_per_volume = 2,
)
print(f'Train dataset: {len(train_ds)} patches')

sample  = train_ds[0]
print(f'image shape : {sample["image"].shape}   ← should be (1, P, P, P)')
print(f'mask shape  : {sample["mask"].shape}    ← should be (1, P, P, P)')
print(f'patient     : {sample["patient_id"]}')

# Sample 8 patches and show how many contain tumour
tumour_counts = []
for i in range(8):
    s = train_ds[i]
    tumour_counts.append(int(s['mask'].sum() > 0))
print(f'\nTumour-containing patches (8 sampled): {sum(tumour_counts)}/8')
print('(Should be ~6-7 due to 80% tumour bias)')

# Visualise one patch
img_np = sample['image'].squeeze().numpy()
msk_np = sample['mask'].squeeze().numpy()
mid    = img_np.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(13, 4), facecolor=BG_COLOR)
axes[0].imshow(img_np[mid].T, cmap='gray', origin='lower')
axes[0].set_title('FLAIR patch — mid slice', color='white')

axes[1].imshow(msk_np[mid].T, cmap='hot', origin='lower', vmin=0, vmax=1)
axes[1].set_title('Tumour mask', color='white')

axes[2].imshow(img_np[mid].T, cmap='gray', origin='lower')
if msk_np.sum() > 0:
    axes[2].contour(msk_np[mid].T, levels=[0.5], colors=[TEAL], linewidths=1.8)
axes[2].set_title('FLAIR + mask contour', color='white')

for ax in axes:
    ax.axis('off')
    ax.set_facecolor(BG_COLOR)

plt.tight_layout()
plt.show()
print('✓ Dataset OK')

---
## Cell 5 — Train both models

> **Kaggle P100/T4**: ~15-25 min per model for 50 epochs at 64³ patches.  
> MLflow runs are logged to `experiments/mlruns/` — open in another terminal with `mlflow ui`.
> To train only one: change `--arch both` → `--arch resunet` or `--arch segresnet`

In [ ]:
import subprocess, sys

# Change --arch to 'resunet' or 'segresnet' to train just one
# Add --epochs 20 for a quick test run
result = subprocess.run(
    [sys.executable, 'train.py', '--arch', 'both'],
    cwd=str(Path().resolve()),   # run from member4_T2f/
)
print(f'\nReturn code: {result.returncode}')
if result.returncode == 0:
    print('✓ Training complete — check checkpoints/ directory')
else:
    print('✗ Training failed — check the output above')

In [ ]:
# ── Plot training curves from MLflow ────────────────────────────────────────
import mlflow

client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name('brats-gli-2024')

if exp:
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="tags.mlflow.runName LIKE 'M4-%'",
        order_by=['start_time DESC'],
        max_results=6,
    )

    arch_colours = {'ResUNet3D': '#5DCAA5', 'SegResNet': '#EF9F27'}
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG_COLOR)

    for run in runs:
        arch_tag = run.data.params.get('architecture', run.data.tags.get('mlflow.runName', 'Unknown'))
        colour   = arch_colours.get(arch_tag, '#aaa')

        train_loss = client.get_metric_history(run.info.run_id, 'train_loss')
        val_dice   = client.get_metric_history(run.info.run_id, 'val_dice')

        if train_loss:
            steps  = [m.step for m in train_loss]
            values = [m.value for m in train_loss]
            axes[0].plot(steps, values, color=colour, label=arch_tag, linewidth=2)

        if val_dice:
            steps  = [m.step for m in val_dice]
            values = [m.value for m in val_dice]
            axes[1].plot(steps, values, color=colour, marker='o',
                         markersize=4, label=arch_tag, linewidth=2)

    for ax, ylabel, title in zip(
        axes,
        ['Loss', 'Dice'],
        ['Training Loss', 'Validation Dice']
    ):
        ax.set_facecolor('#161B22')
        ax.tick_params(colors='white')
        ax.set_xlabel('Epoch', color='white')
        ax.set_ylabel(ylabel, color='white')
        ax.set_title(title, color='white')
        ax.spines[:].set_color('#444')
        ax.legend(facecolor='#222', labelcolor='white')
        ax.grid(alpha=0.2, color='#555')
        if ylabel == 'Dice':
            ax.axhline(0.70, color='#FF4B4B', linestyle='--', alpha=0.5, label='Target 0.70')

    fig.suptitle('M4 Training Curves — ResUNet3D vs SegResNet', color='white', fontsize=13)
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / 'T2F' / 'figures' / 'M4_training_curves.png',
                dpi=120, bbox_inches='tight', facecolor=BG_COLOR)
    plt.show()
else:
    print('No MLflow experiment found yet — run Cell 5 first')

---
## Cell 6 — Evaluate on test set

In [ ]:
result = subprocess.run(
    [sys.executable, 'evaluate.py', '--arch', 'both'],
    cwd=str(Path().resolve()),
)
print(f'Return code: {result.returncode}')

In [ ]:
import pandas as pd

full_csv = RESULTS_ROOT / 'T2F' / 'tables' / 'M4_test_full.csv'
if full_csv.exists():
    df = pd.read_csv(full_csv)
    print('M4 TEST RESULTS\n')
    print(df[['Architecture', 'Dice_WT', 'IoU_WT', 'HD95_WT', 'Val_Dice']].to_string(index=False))
    
    # Bar chart
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), facecolor=BG_COLOR)
    metrics  = ['Dice_WT', 'IoU_WT', 'HD95_WT']
    targets  = [0.70, 0.55, 10.0]
    y_labels = ['Dice WT', 'IoU WT', 'HD95 (mm, lower=better)']
    bar_cols = ['#5DCAA5', '#EF9F27']

    for ax, metric, target, ylabel in zip(axes, metrics, targets, y_labels):
        ax.set_facecolor('#161B22')
        bars = ax.bar(df['Architecture'], df[metric],
                      color=bar_cols[:len(df)], edgecolor='#444', linewidth=0.8)
        ax.axhline(target, color='#FF4B4B', linestyle='--', alpha=0.7,
                   label=f'Target: {target}')
        for bar, val in zip(bars, df[metric]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', color='white', fontsize=9)
        ax.set_title(ylabel, color='white')
        ax.tick_params(colors='white')
        ax.spines[:].set_color('#444')
        ax.legend(facecolor='#222', labelcolor='white', fontsize=8)

    fig.suptitle('M4 Test Metrics — ResUNet3D vs SegResNet', color='white', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / 'T2F' / 'figures' / 'M4_test_metrics.png',
                dpi=120, bbox_inches='tight', facecolor=BG_COLOR)
    plt.show()
else:
    print('Run Cell 6 evaluate.py first')

---
## Cell 7 — Grad-CAM XAI visualisation
> Produces overlays for 3 test patients × both architectures.  
> Add `--shap_bg 20` to also run SHAP (takes ~30 min on CPU).

In [ ]:
# Grad-CAM only (fast, ~2 min)
result = subprocess.run(
    [sys.executable, 'xai_analysis.py', '--n_patients', '3'],
    cwd=str(Path().resolve()),
)
print(f'Return code: {result.returncode}')

In [ ]:
# Display all generated Grad-CAM figures inline
from IPython.display import display, Image as IPImage
import glob

figures_dir = RESULTS_ROOT / 'T2F' / 'figures'
cam_figs    = sorted(figures_dir.glob('M4_gradcam_*_combined.png'))
comp_figs   = sorted(figures_dir.glob('M4_arch_comparison_*.png'))

print(f'Found {len(cam_figs)} Grad-CAM overlays and {len(comp_figs)} comparison figures\n')

for f in cam_figs[:3]:
    print(f.name)
    display(IPImage(str(f), width=900))

print('--- Architecture Comparisons ---')
for f in comp_figs[:3]:
    print(f.name)
    display(IPImage(str(f), width=900))

In [ ]:
# OPTIONAL — run SHAP overnight (remove --shap_bg 0 to use default 20 samples)
# This cell takes ~20-40 minutes on CPU. Skip if short on time.

# Uncomment to run:
# result = subprocess.run(
#     [sys.executable, 'xai_analysis.py', '--n_patients', '2', '--shap_bg', '20'],
#     cwd=str(Path().resolve()),
# )
# print(f'SHAP done. Return code: {result.returncode}')
print('SHAP cell — uncomment to run')

---
## Cell 8 — Launch Gradio demo app

> This cell starts an interactive web app.  
> On Kaggle: a **public shareable link** appears (valid 72 h).  
> Share the link with your supervisor or teammates — they can explore predictions in a browser with no setup.

In [ ]:
# Install gradio if not already done
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'], check=True)

# Launch the app
# The public URL will be printed below — copy and share it
%run app.py